# AgeLens — 01 Data Ingestion

This notebook ingests and validates:

1. NHANES 2015–2016 and 2017–2018 biomarker source files.
2. The corresponding 2019 public-use linked mortality `.dat` files, when present.

## Placement

```text
nhanes/
└── notebooks/
    ├── 00_setup_agelens.ipynb
    └── 01_data_ingestion.ipynb
```

## Required NHANES XPT files

```text
nhanes/data/raw/2015_2016/
├── DEMO_I.XPT
├── BIOPRO_I.XPT
├── GLU_I.XPT
├── HSCRP_I.XPT
└── CBC_I.XPT
```

```text
nhanes/data/raw/2017_2018/
├── DEMO_J.XPT
├── BIOPRO_J.XPT
├── GLU_J.XPT
├── HSCRP_J.XPT
└── CBC_J.XPT
```

## Mortality files

```text
nhanes/data/raw/mortality/2019_public/
├── NHANES_2015_2016_MORT_2019_PUBLIC.dat
└── NHANES_2017_2018_MORT_2019_PUBLIC.dat
```

## Important design choices

- `LBXGLU` is read from `GLU_I/J`.
- `WTSAF2YR` is also read from `GLU_I/J`, not from `DEMO_I/J`.
- `LBXSGL` from `BIOPRO_I/J` is deliberately excluded from the primary pipeline.
- Laboratory files are left-joined to demographics so raw missingness remains visible.
- Mortality files are parsed and saved separately.
- Mortality outcomes are **not** merged into the biomarker dataset in this notebook.
- No bridging, unit conversion, complete-case filtering, imputation, formula calculation, or survival analysis occurs here.


## XPORT zero normalization

The ingestion layer converts only the exact IBM zero sentinel `5.397605346934028e-79` to `0.0` immediately after each XPT read. Every replacement is recorded in `01_xpt_ibm_zero_sentinel_audit.csv`; no general near-zero threshold is applied.


In [1]:
from __future__ import annotations

from datetime import datetime, timezone
from hashlib import sha256
from pathlib import Path
from typing import Any

import json
import numpy as np
import pandas as pd

PROJECT_FOLDER_NAME = "nhanes"

# The biomarker pipeline should still run when mortality files are absent.
# Set True only when mortality ingestion must be mandatory.
MORTALITY_REQUIRED = False

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 170)

print(f"pandas: {pd.__version__}")
print(f"Current working directory: {Path.cwd().resolve()}")


pandas: 2.3.3
Current working directory: <PROJECT_ROOT>\notebooks


## 1. Locate the project and load configuration

In [2]:
def find_project_root(folder_name: str = PROJECT_FOLDER_NAME) -> Path:
    current = Path.cwd().resolve()

    for candidate in [current, *current.parents]:
        if candidate.name.lower() == folder_name.lower():
            return candidate

    raise FileNotFoundError(
        f"Could not find a parent folder named '{folder_name}'. "
        "Run this notebook from inside the nhanes project."
    )


PROJECT_ROOT = find_project_root()
CONFIG_PATH = PROJECT_ROOT / "configs" / "agelens_config.json"

if not CONFIG_PATH.exists():
    raise FileNotFoundError(
        f"Configuration not found: {CONFIG_PATH}\n"
        "Run notebooks/00_setup_agelens.ipynb first."
    )

CONFIG: dict[str, Any] = json.loads(CONFIG_PATH.read_text(encoding="utf-8"))

required_config_sections = {
    "project",
    "governance",
    "nhanes",
    "mortality",
    "source_variables",
    "pipeline",
    "paths",
}
missing_sections = required_config_sections - set(CONFIG)
if missing_sections:
    raise ValueError(
        "The configuration is older than the required setup schema. "
        f"Missing sections: {sorted(missing_sections)}. "
        "Run the updated 00_setup_agelens.ipynb with "
        "OVERWRITE_CONFIG = True after reviewing the existing config."
    )

RAW_ROOT = PROJECT_ROOT / CONFIG["paths"]["raw_data"]
MORTALITY_ROOT = PROJECT_ROOT / CONFIG["paths"]["raw_mortality"]
INTERIM_ROOT = PROJECT_ROOT / CONFIG["paths"]["interim_data"]
TABLES_ROOT = PROJECT_ROOT / CONFIG["paths"]["tables"]
LOGS_ROOT = PROJECT_ROOT / CONFIG["paths"]["logs"]

for path in [INTERIM_ROOT, TABLES_ROOT, LOGS_ROOT]:
    path.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Config: {CONFIG_PATH}")
print(f"Mortality directory: {MORTALITY_ROOT}")


Project root: <PROJECT_ROOT>
Config: <PROJECT_ROOT>\configs\agelens_config.json
Mortality directory: <PROJECT_ROOT>\data\raw\mortality\2019_public


## 2. Controlled source-file specification

In [3]:
COMPONENT_SPECS = {
    "demographics": {
        "required": [
            "SEQN",
            "RIDAGEYR",
            "RIAGENDR",
            "RIDRETH3",
            "SDMVPSU",
            "SDMVSTRA",
        ],
        "optional": [
            "WTINT2YR",
            "WTMEC2YR",
        ],
    },
    "biochemistry": {
        "required": [
            "SEQN",
            "LBXSAL",
            "LBXSCR",
            "LBXSAPSI",
        ],
        "optional": [],
    },
    "fasting_glucose": {
        "required": [
            "SEQN",
            "WTSAF2YR",
            "LBXGLU",
        ],
        "optional": [
            "LBDGLUSI",
        ],
    },
    "hscrp": {
        "required": [
            "SEQN",
            "LBXHSCRP",
        ],
        "optional": [],
    },
    "cbc": {
        "required": [
            "SEQN",
            "LBXLYPCT",
            "LBXMCVSI",
            "LBXRDW",
            "LBXWBCSI",
        ],
        "optional": [],
    },
}

PRIMARY_SOURCE_VARIABLES = {
    key: value
    for key, value in CONFIG["source_variables"].items()
    if key != "biopro_glucose_not_for_primary_pipeline"
}

assert PRIMARY_SOURCE_VARIABLES["glucose"] == "LBXGLU"
assert CONFIG["nhanes"]["survey_design"]["weight_variable"] == "WTSAF2YR"

print("Controlled components:")
for component_name, spec in COMPONENT_SPECS.items():
    print(
        f"  - {component_name}: "
        f"{len(spec['required'])} required, "
        f"{len(spec['optional'])} optional"
    )


Controlled components:
  - demographics: 6 required, 2 optional
  - biochemistry: 4 required, 0 optional
  - fasting_glucose: 3 required, 1 optional
  - hscrp: 2 required, 0 optional
  - cbc: 5 required, 0 optional


## 3. Utility and validation functions

In [4]:
def file_sha256(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = sha256()
    with path.open("rb") as handle:
        while chunk := handle.read(chunk_size):
            digest.update(chunk)
    return digest.hexdigest()


def normalize_seqn(frame: pd.DataFrame, source_name: str) -> pd.DataFrame:
    frame = frame.copy()

    if "SEQN" not in frame.columns:
        raise ValueError(f"{source_name} does not contain SEQN.")

    seqn_numeric = pd.to_numeric(frame["SEQN"], errors="raise")

    non_missing = seqn_numeric.notna()
    if not np.all(np.equal(np.mod(seqn_numeric[non_missing], 1), 0)):
        raise ValueError(f"{source_name} contains non-integer SEQN values.")

    frame["SEQN"] = seqn_numeric.astype("Int64")
    return frame


# pandas read_sas/XPORT may decode true IBM-format zeros as the
# smallest positive IBM float. Normalize only this exact sentinel.
XPT_IBM_ZERO_SENTINEL = np.float64(
    5.397605346934028e-79
)
xpt_zero_sentinel_audit_records: list[
    dict[str, Any]
] = []


def normalize_xpt_ibm_zero_sentinel(
    frame: pd.DataFrame,
    *,
    source_name: str,
) -> pd.DataFrame:
    frame = frame.copy()

    numeric_columns = frame.select_dtypes(
        include=[np.number]
    ).columns.tolist()

    for column in numeric_columns:
        values = pd.to_numeric(
            frame[column],
            errors="coerce",
        )

        sentinel_mask = values.eq(
            XPT_IBM_ZERO_SENTINEL
        )
        replacement_count = int(
            sentinel_mask.sum()
        )

        if replacement_count:
            xpt_zero_sentinel_audit_records.append(
                {
                    "file": source_name,
                    "column": column,
                    "sentinel_value": float(
                        XPT_IBM_ZERO_SENTINEL
                    ),
                    "replacement_count": (
                        replacement_count
                    ),
                    "replacement_value": 0.0,
                }
            )
            frame.loc[sentinel_mask, column] = 0.0

    remaining = 0
    for column in numeric_columns:
        remaining += int(
            pd.to_numeric(
                frame[column],
                errors="coerce",
            ).eq(XPT_IBM_ZERO_SENTINEL).sum()
        )

    if remaining:
        raise RuntimeError(
            f"{source_name} still contains {remaining} "
            "IBM-zero sentinel values after normalization."
        )

    return frame


def read_xpt(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(
            f"Required NHANES file not found: {path}"
        )

    try:
        frame = pd.read_sas(
            path,
            format="xport",
            encoding="utf-8",
        )
    except UnicodeDecodeError:
        frame = pd.read_sas(
            path,
            format="xport",
            encoding="latin-1",
        )
    except Exception as exc:
        raise RuntimeError(
            f"Failed to read XPT file: {path}"
        ) from exc

    frame.columns = [
        str(column).strip().upper()
        for column in frame.columns
    ]

    frame = normalize_xpt_ibm_zero_sentinel(
        frame,
        source_name=path.name,
    )

    return normalize_seqn(frame, path.name)


def validate_and_select_component(
    frame: pd.DataFrame,
    *,
    component_name: str,
    file_path: Path,
) -> tuple[pd.DataFrame, dict[str, Any]]:
    spec = COMPONENT_SPECS[component_name]
    required = spec["required"]
    optional = [column for column in spec["optional"] if column in frame.columns]

    missing = [column for column in required if column not in frame.columns]
    if missing:
        raise ValueError(
            f"{file_path.name} is missing required columns "
            f"for {component_name}: {missing}"
        )

    if frame["SEQN"].isna().any():
        raise ValueError(f"{file_path.name} contains missing SEQN values.")

    duplicate_count = int(frame["SEQN"].duplicated().sum())
    if duplicate_count:
        examples = (
            frame.loc[frame["SEQN"].duplicated(keep=False), "SEQN"]
            .head(10)
            .tolist()
        )
        raise ValueError(
            f"{file_path.name} contains {duplicate_count} duplicate SEQN rows. "
            f"Examples: {examples}"
        )

    selected_columns = required + optional
    selected = frame.loc[:, selected_columns].copy()

    audit = {
        "component": component_name,
        "file": file_path.name,
        "file_size_bytes": file_path.stat().st_size,
        "sha256": file_sha256(file_path),
        "rows": int(len(frame)),
        "selected_columns": len(selected_columns),
        "unique_seqn": int(frame["SEQN"].nunique()),
        "missing_seqn": int(frame["SEQN"].isna().sum()),
        "duplicate_seqn": duplicate_count,
    }

    return selected, audit


def safe_one_to_one_merge(
    left: pd.DataFrame,
    right: pd.DataFrame,
    *,
    component_name: str,
) -> pd.DataFrame:
    overlapping = sorted((set(left.columns) & set(right.columns)) - {"SEQN"})
    if overlapping:
        raise ValueError(
            f"Unexpected overlapping columns while merging "
            f"{component_name}: {overlapping}"
        )

    return left.merge(
        right,
        on="SEQN",
        how="left",
        validate="one_to_one",
        sort=False,
    )


## 4. Verify configured NHANES XPT files

In [5]:
missing_xpt_files: list[Path] = []
configured_xpt_files: list[Path] = []

for cycle, cycle_config in CONFIG["nhanes"]["cycles"].items():
    cycle_directory = RAW_ROOT / cycle

    for component_name in COMPONENT_SPECS:
        file_path = cycle_directory / cycle_config[component_name]
        configured_xpt_files.append(file_path)

        if not file_path.exists():
            missing_xpt_files.append(file_path)

if missing_xpt_files:
    formatted = "\n".join(f"  - {path}" for path in missing_xpt_files)
    raise FileNotFoundError(
        "The following required NHANES XPT files are missing:\n"
        f"{formatted}\n\n"
        "GLU_I.XPT and GLU_J.XPT are listed on the CDC NHANES laboratory "
        "data pages under 'Plasma Fasting Glucose'."
    )

print("✅ All configured NHANES XPT files are present.")
for path in configured_xpt_files:
    print(f"  - {path.relative_to(PROJECT_ROOT)}")


✅ All configured NHANES XPT files are present.
  - data\raw\2015_2016\DEMO_I.XPT
  - data\raw\2015_2016\BIOPRO_I.XPT
  - data\raw\2015_2016\GLU_I.XPT
  - data\raw\2015_2016\HSCRP_I.XPT
  - data\raw\2015_2016\CBC_I.XPT
  - data\raw\2017_2018\DEMO_J.XPT
  - data\raw\2017_2018\BIOPRO_J.XPT
  - data\raw\2017_2018\GLU_J.XPT
  - data\raw\2017_2018\HSCRP_J.XPT
  - data\raw\2017_2018\CBC_J.XPT


## 5. Ingest and merge each NHANES cycle

Demographics is the merge base. Laboratory components are left-joined so structural and incidental missingness remain visible.


In [6]:
cycle_frames: dict[str, pd.DataFrame] = {}
component_audit_records: list[dict[str, Any]] = []
merge_audit_records: list[dict[str, Any]] = []

for cycle, cycle_config in CONFIG["nhanes"]["cycles"].items():
    print(f"\nLoading cycle: {cycle}")
    cycle_directory = RAW_ROOT / cycle
    component_frames: dict[str, pd.DataFrame] = {}

    for component_name in COMPONENT_SPECS:
        file_path = cycle_directory / cycle_config[component_name]
        raw_frame = read_xpt(file_path)

        selected_frame, audit = validate_and_select_component(
            raw_frame,
            component_name=component_name,
            file_path=file_path,
        )
        audit["cycle"] = cycle

        component_audit_records.append(audit)
        component_frames[component_name] = selected_frame

        print(
            f"  {component_name:<18} "
            f"rows={len(selected_frame):>6,} "
            f"columns={len(selected_frame.columns):>2}"
        )

    merged = component_frames["demographics"].copy()
    demographic_rows = len(merged)

    for component_name in [
        "biochemistry",
        "fasting_glucose",
        "hscrp",
        "cbc",
    ]:
        before_rows = len(merged)
        matched_count = int(
            merged["SEQN"].isin(
                component_frames[component_name]["SEQN"]
            ).sum()
        )

        merged = safe_one_to_one_merge(
            merged,
            component_frames[component_name],
            component_name=component_name,
        )

        if len(merged) != before_rows:
            raise RuntimeError(
                f"Row count changed while merging {component_name} "
                f"for cycle {cycle}."
            )

        merge_audit_records.append(
            {
                "cycle": cycle,
                "component": component_name,
                "base_rows": before_rows,
                "matched_seqn": matched_count,
                "unmatched_seqn": before_rows - matched_count,
                "rows_after_merge": len(merged),
            }
        )

    cycle_column = CONFIG["nhanes"]["cycle_column"]
    merged[cycle_column] = cycle

    merged["age_topcoded"] = (
        merged[CONFIG["nhanes"]["age_variable"]]
        == CONFIG["nhanes"]["age_topcode_value"]
    )

    merged["fasting_subsample_record_present"] = (
        merged["WTSAF2YR"].notna()
    )
    merged["fasting_weight_positive"] = (
        merged["WTSAF2YR"].fillna(0) > 0
    )
    merged["WTSAF4YR"] = (
        merged["WTSAF2YR"] / 2.0
    )

    sentinel_columns = [
        column
        for column in merged.select_dtypes(
            include=[np.number]
        ).columns
        if pd.to_numeric(
            merged[column],
            errors="coerce",
        ).eq(XPT_IBM_ZERO_SENTINEL).any()
    ]

    if sentinel_columns:
        raise RuntimeError(
            f"Cycle {cycle} still contains IBM-zero "
            f"sentinel values in: {sentinel_columns}"
        )

    print(
        "  fasting GLU records="
        f"{int(merged['fasting_subsample_record_present'].sum()):,}; "
        "positive fasting weights="
        f"{int(merged['fasting_weight_positive'].sum()):,}; "
        "zero fasting weights="
        f"{int(merged['WTSAF2YR'].eq(0).sum()):,}"
    )

    if "LBXSGL" in merged.columns:
        raise RuntimeError(
            "LBXSGL entered the controlled primary dataset unexpectedly."
        )

    front = [
        "SEQN",
        cycle_column,
        "RIDAGEYR",
        "age_topcoded",
        "RIAGENDR",
        "RIDRETH3",
        "WTSAF2YR",
        "WTSAF4YR",
        "fasting_subsample_record_present",
        "fasting_weight_positive",
        "SDMVSTRA",
        "SDMVPSU",
    ]
    remaining = [
        column for column in merged.columns
        if column not in front
    ]
    merged = merged.loc[:, front + remaining]

    if merged["SEQN"].duplicated().any():
        raise RuntimeError(f"Cycle {cycle} contains duplicate SEQN.")

    if len(merged) != demographic_rows:
        raise RuntimeError(
            f"Demographic row count was not preserved for {cycle}."
        )

    weight_mask = merged["WTSAF2YR"].notna()
    if not np.allclose(
        merged.loc[weight_mask, "WTSAF4YR"],
        merged.loc[weight_mask, "WTSAF2YR"] / 2.0,
    ):
        raise RuntimeError(f"Pooled fasting weight check failed for {cycle}.")

    cycle_frames[cycle] = merged
    print(f"  merged rows={len(merged):,}")



Loading cycle: 2015_2016
  demographics       rows= 9,971 columns= 8
  biochemistry       rows= 6,744 columns= 4
  fasting_glucose    rows= 3,191 columns= 4
  hscrp              rows= 9,165 columns= 2
  cbc                rows= 9,165 columns= 5
  fasting GLU records=3,191; positive fasting weights=2,743; zero fasting weights=448
  merged rows=9,971

Loading cycle: 2017_2018
  demographics       rows= 9,254 columns= 8
  biochemistry       rows= 6,401 columns= 4
  fasting_glucose    rows= 3,036 columns= 4
  hscrp              rows= 8,366 columns= 2
  cbc                rows= 8,366 columns= 5
  fasting GLU records=3,036; positive fasting weights=2,711; zero fasting weights=325
  merged rows=9,254


## 6. Combine cycles and verify the biomarker dataset

In [7]:
cycle_column = CONFIG["nhanes"]["cycle_column"]

combined_biomarker = pd.concat(
    [
        cycle_frames[cycle]
        for cycle in CONFIG["nhanes"]["cycles"]
    ],
    ignore_index=True,
    sort=False,
)

if combined_biomarker.duplicated([cycle_column, "SEQN"]).any():
    raise RuntimeError(
        "Duplicate cycle + SEQN combinations found after concatenation."
    )

required_final_columns = {
    "SEQN",
    cycle_column,
    "RIDAGEYR",
    "age_topcoded",
    "WTSAF2YR",
    "WTSAF4YR",
    "SDMVSTRA",
    "SDMVPSU",
    *PRIMARY_SOURCE_VARIABLES.values(),
}

missing_final_columns = sorted(
    required_final_columns - set(combined_biomarker.columns)
)
if missing_final_columns:
    raise RuntimeError(
        "Combined biomarker dataset is missing required columns: "
        f"{missing_final_columns}"
    )

assert "LBXGLU" in combined_biomarker.columns
assert "LBXSGL" not in combined_biomarker.columns
assert set(combined_biomarker[cycle_column].unique()) == set(
    CONFIG["nhanes"]["cycles"]
)

print("✅ Combined biomarker-ingestion checks passed.")
print(f"Rows: {len(combined_biomarker):,}")
print(f"Columns: {len(combined_biomarker.columns)}")
print("Participant-level preview omitted from the public notebook.")


✅ Combined biomarker-ingestion checks passed.
Rows: 19,225
Columns: 24
Participant-level preview omitted from the public notebook.


## 7. Public-use mortality parser

In [8]:
# Official 2019 public-use NHANES linked-mortality layout.
# Python colspec endpoints are zero-based and end-exclusive.
MORTALITY_COLSPECS = [
    (0, 6),    # SEQN: positions 1-6
    (14, 15),  # ELIGSTAT: position 15
    (15, 16),  # MORTSTAT: position 16
    (16, 19),  # UCOD_LEADING: positions 17-19
    (19, 20),  # DIABETES: position 20
    (20, 21),  # HYPERTEN: position 21
    (21, 22),  # DODQTR: position 22; NHIS only
    (22, 26),  # DODYEAR: positions 23-26; NHIS only
    (42, 45),  # PERMTH_INT: positions 43-45
    (45, 48),  # PERMTH_EXM: positions 46-48
]

MORTALITY_NAMES = [
    "SEQN",
    "ELIGSTAT",
    "MORTSTAT",
    "UCOD_LEADING",
    "DIABETES",
    "HYPERTEN",
    "DODQTR",
    "DODYEAR",
    "PERMTH_INT",
    "PERMTH_EXM",
]

MORTALITY_NUMERIC_COLUMNS = [
    "SEQN",
    "ELIGSTAT",
    "MORTSTAT",
    "DIABETES",
    "HYPERTEN",
    "DODQTR",
    "DODYEAR",
    "PERMTH_INT",
    "PERMTH_EXM",
]


def read_nhanes_mortality_2019(
    path: Path,
    *,
    cycle: str,
) -> tuple[pd.DataFrame, dict[str, Any]]:
    if not path.exists():
        raise FileNotFoundError(f"Mortality file not found: {path}")

    raw_line_lengths = []
    with path.open("r", encoding="ascii", errors="strict") as handle:
        for line in handle:
            raw_line_lengths.append(len(line.rstrip("\r\n")))

    frame = pd.read_fwf(
        path,
        colspecs=MORTALITY_COLSPECS,
        names=MORTALITY_NAMES,
        dtype=str,
        keep_default_na=False,
    )

    for column in frame.columns:
        frame[column] = (
            frame[column]
            .astype("string")
            .str.strip()
            .replace({"": pd.NA, ".": pd.NA})
        )

    for column in MORTALITY_NUMERIC_COLUMNS:
        frame[column] = pd.to_numeric(
            frame[column],
            errors="raise",
        ).astype("Int64")

    frame["UCOD_LEADING"] = frame["UCOD_LEADING"].astype("string")
    frame[cycle_column] = cycle
    frame["MORTALITY_RELEASE"] = CONFIG["mortality"]["release"]

    if frame["SEQN"].isna().any():
        raise ValueError(f"{path.name} contains missing SEQN.")

    if frame["SEQN"].duplicated().any():
        raise ValueError(f"{path.name} contains duplicate SEQN.")

    allowed_eligstat = {1, 2, 3}
    observed_eligstat = set(
        frame["ELIGSTAT"].dropna().astype(int).unique()
    )
    if not observed_eligstat.issubset(allowed_eligstat):
        raise ValueError(
            f"{path.name} contains invalid ELIGSTAT values: "
            f"{sorted(observed_eligstat - allowed_eligstat)}"
        )

    observed_mortstat = set(
        frame["MORTSTAT"].dropna().astype(int).unique()
    )
    if not observed_mortstat.issubset({0, 1}):
        raise ValueError(
            f"{path.name} contains invalid MORTSTAT values."
        )

    eligible = frame["ELIGSTAT"].eq(1)
    if frame.loc[eligible, "MORTSTAT"].isna().any():
        raise ValueError(
            f"{path.name} has eligible participants with missing MORTSTAT."
        )
    if frame.loc[~eligible, "MORTSTAT"].notna().any():
        raise ValueError(
            f"{path.name} has mortality status for ineligible/minor records."
        )

    for column in ["DIABETES", "HYPERTEN"]:
        observed = set(frame[column].dropna().astype(int).unique())
        if not observed.issubset({0, 1}):
            raise ValueError(
                f"{path.name} contains invalid {column} values."
            )

    allowed_ucod = {"001", "002", "010"}
    observed_ucod = set(frame["UCOD_LEADING"].dropna().unique())
    if not observed_ucod.issubset(allowed_ucod):
        raise ValueError(
            f"{path.name} contains unexpected UCOD_LEADING values: "
            f"{sorted(observed_ucod - allowed_ucod)}"
        )

    for column in ["PERMTH_INT", "PERMTH_EXM"]:
        valid = frame[column].dropna()
        if not valid.between(0, 374).all():
            raise ValueError(
                f"{path.name} contains out-of-range {column} values."
            )

    if frame["DODQTR"].notna().any() or frame["DODYEAR"].notna().any():
        raise ValueError(
            f"{path.name} contains NHIS-only DODQTR/DODYEAR values."
        )

    audit = {
        "cycle": cycle,
        "file": path.name,
        "file_size_bytes": path.stat().st_size,
        "sha256": file_sha256(path),
        "rows": int(len(frame)),
        "unique_seqn": int(frame["SEQN"].nunique()),
        "eligible": int(frame["ELIGSTAT"].eq(1).sum()),
        "under_18_public_release_unavailable": int(
            frame["ELIGSTAT"].eq(2).sum()
        ),
        "linkage_ineligible": int(frame["ELIGSTAT"].eq(3).sum()),
        "assumed_alive": int(frame["MORTSTAT"].eq(0).sum()),
        "assumed_deceased": int(frame["MORTSTAT"].eq(1).sum()),
        "minimum_line_length": min(raw_line_lengths),
        "maximum_line_length": max(raw_line_lengths),
    }

    return frame, audit


## 8. Ingest mortality files separately

Mortality ingestion is optional unless `MORTALITY_REQUIRED = True`. When both files are available, they are validated and saved as a separate dataset.


In [9]:
mortality_paths = {
    cycle: MORTALITY_ROOT / filename
    for cycle, filename in CONFIG["mortality"]["files"].items()
}

missing_mortality_files = [
    path for path in mortality_paths.values()
    if not path.exists()
]

mortality_frames: dict[str, pd.DataFrame] = {}
mortality_audit_records: list[dict[str, Any]] = []
linkage_audit_records: list[dict[str, Any]] = []

if missing_mortality_files and MORTALITY_REQUIRED:
    formatted = "\n".join(
        f"  - {path}" for path in missing_mortality_files
    )
    raise FileNotFoundError(
        "Mortality ingestion was marked as required, but files are missing:\n"
        f"{formatted}"
    )

if missing_mortality_files:
    print("⚠️ Mortality ingestion skipped because one or more files are absent:")
    for path in missing_mortality_files:
        print(f"  - {path}")
else:
    for cycle, path in mortality_paths.items():
        mortality_frame, mortality_audit = read_nhanes_mortality_2019(
            path,
            cycle=cycle,
        )

        mortality_frames[cycle] = mortality_frame
        mortality_audit_records.append(mortality_audit)

        demographic_seqn = set(cycle_frames[cycle]["SEQN"].dropna().astype(int))
        mortality_seqn = set(mortality_frame["SEQN"].dropna().astype(int))

        linkage_audit = {
            "cycle": cycle,
            "demographic_rows": len(cycle_frames[cycle]),
            "mortality_rows": len(mortality_frame),
            "matched_seqn": len(demographic_seqn & mortality_seqn),
            "demographic_only_seqn": len(demographic_seqn - mortality_seqn),
            "mortality_only_seqn": len(mortality_seqn - demographic_seqn),
        }
        linkage_audit_records.append(linkage_audit)

        if demographic_seqn != mortality_seqn:
            raise RuntimeError(
                f"SEQN coverage differs between DEMO and mortality for {cycle}. "
                f"Audit: {linkage_audit}"
            )

        print(
            f"✅ {cycle} mortality: "
            f"{len(mortality_frame):,} rows, "
            f"{mortality_audit['assumed_deceased']:,} deceased"
        )

    combined_mortality = pd.concat(
        [
            mortality_frames[cycle]
            for cycle in CONFIG["nhanes"]["cycles"]
        ],
        ignore_index=True,
        sort=False,
    )

    if combined_mortality.duplicated([cycle_column, "SEQN"]).any():
        raise RuntimeError(
            "Duplicate cycle + SEQN combinations in mortality dataset."
        )

    display(pd.DataFrame(mortality_audit_records))


✅ 2015_2016 mortality: 9,971 rows, 276 deceased
✅ 2017_2018 mortality: 9,254 rows, 145 deceased


,cycle,file,file_size_bytes,sha256,rows,unique_seqn,eligible,under_18_public_release_unavailable,linkage_ineligible,assumed_alive,assumed_deceased,minimum_line_length,maximum_line_length
0,2015_2016,NHANES_2015_2016_MORT_2019_PUBLIC.dat,484288,f392baaf9c20a4c75bc9936775ba956e64528fa5ada510...,9971,9971,5974,3979,18,5698,276,46,47
1,2017_2018,NHANES_2017_2018_MORT_2019_PUBLIC.dat,449623,42989d4fe35754d696b770f26553b0a309c110a9e26ab0...,9254,9254,5809,3398,47,5664,145,46,47


## 9. Raw biomarker availability audit

In [10]:
biomarker_columns = [
    "LBXSAL",
    "LBXSCR",
    "LBXGLU",
    "LBXHSCRP",
    "LBXLYPCT",
    "LBXMCVSI",
    "LBXRDW",
    "LBXSAPSI",
    "LBXWBCSI",
]

availability_records = []

for cycle, frame in cycle_frames.items():
    for column in biomarker_columns:
        availability_records.append(
            {
                "cycle": cycle,
                "variable": column,
                "rows": len(frame),
                "non_missing": int(frame[column].notna().sum()),
                "missing": int(frame[column].isna().sum()),
                "percent_non_missing": float(
                    frame[column].notna().mean() * 100
                ),
            }
        )

availability_audit = pd.DataFrame(availability_records)

display(
    availability_audit.pivot(
        index="variable",
        columns="cycle",
        values="percent_non_missing",
    ).round(2)
)


cycle,2015_2016,2017_2018
variable,,
LBXGLU,29.81,31.24
LBXHSCRP,78.90,78.34
LBXLYPCT,81.40,81.29
LBXMCVSI,81.41,81.35
LBXRDW,81.41,81.35
LBXSAL,62.74,63.81
LBXSAPSI,62.73,63.79
LBXSCR,62.73,63.79
LBXWBCSI,81.41,81.35


## 10. Save interim datasets and audit artifacts

Biomarker outputs remain in native NHANES units. Mortality outputs remain separate. Neither output is a final analytic dataset.


In [11]:
written_files: list[Path] = []

# Biomarker outputs
for cycle, frame in cycle_frames.items():
    output = (
        INTERIM_ROOT
        / f"nhanes_{cycle}_ingested_native.parquet"
    )
    frame.to_parquet(output, index=False)
    written_files.append(output)

combined_biomarker_output = (
    INTERIM_ROOT
    / "nhanes_2015_2018_ingested_native.parquet"
)
combined_biomarker.to_parquet(
    combined_biomarker_output,
    index=False,
)
written_files.append(combined_biomarker_output)

# Core audits
component_audit = pd.DataFrame(component_audit_records)
merge_audit = pd.DataFrame(merge_audit_records)

component_audit_path = (
    TABLES_ROOT / "01_component_ingestion_audit.csv"
)
merge_audit_path = TABLES_ROOT / "01_merge_audit.csv"
availability_audit_path = (
    TABLES_ROOT / "01_variable_availability_audit.csv"
)

component_audit.to_csv(component_audit_path, index=False)
merge_audit.to_csv(merge_audit_path, index=False)
availability_audit.to_csv(
    availability_audit_path,
    index=False,
)

xpt_zero_sentinel_audit = pd.DataFrame(
    xpt_zero_sentinel_audit_records,
    columns=[
        "file",
        "column",
        "sentinel_value",
        "replacement_count",
        "replacement_value",
    ],
)

xpt_zero_sentinel_audit_path = (
    TABLES_ROOT
    / "01_xpt_ibm_zero_sentinel_audit.csv"
)
xpt_zero_sentinel_audit.to_csv(
    xpt_zero_sentinel_audit_path,
    index=False,
)

total_sentinel_replacements = (
    int(
        xpt_zero_sentinel_audit[
            "replacement_count"
        ].sum()
    )
    if not xpt_zero_sentinel_audit.empty
    else 0
)

print(
    "XPT IBM-zero sentinel replacements:",
    total_sentinel_replacements,
)
display(xpt_zero_sentinel_audit)

written_files.extend(
    [
        component_audit_path,
        merge_audit_path,
        availability_audit_path,
        xpt_zero_sentinel_audit_path,
    ]
)

# Mortality outputs, only when successfully ingested
mortality_ingested = bool(mortality_frames)

if mortality_ingested:
    for cycle, frame in mortality_frames.items():
        output = (
            INTERIM_ROOT
            / f"nhanes_{cycle}_mortality_2019_public.parquet"
        )
        frame.to_parquet(output, index=False)
        written_files.append(output)

    combined_mortality_output = (
        INTERIM_ROOT
        / "nhanes_2015_2018_mortality_2019_public.parquet"
    )
    combined_mortality.to_parquet(
        combined_mortality_output,
        index=False,
    )
    written_files.append(combined_mortality_output)

    mortality_audit_path = (
        TABLES_ROOT / "01_mortality_ingestion_audit.csv"
    )
    linkage_audit_path = (
        TABLES_ROOT / "01_mortality_linkage_coverage_audit.csv"
    )

    pd.DataFrame(mortality_audit_records).to_csv(
        mortality_audit_path,
        index=False,
    )
    pd.DataFrame(linkage_audit_records).to_csv(
        linkage_audit_path,
        index=False,
    )

    written_files.extend(
        [mortality_audit_path, linkage_audit_path]
    )

metadata = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "notebook": "01_data_ingestion.ipynb",
    "config_schema_version": CONFIG["project"]["config_schema_version"],
    "source_cycles": list(CONFIG["nhanes"]["cycles"]),
    "biomarker_row_count": int(len(combined_biomarker)),
    "biomarker_column_count": int(len(combined_biomarker.columns)),
    "mortality_ingested": mortality_ingested,
    "mortality_release": (
        CONFIG["mortality"]["release"]
        if mortality_ingested else None
    ),
    "native_units": True,
    "xpt_ibm_zero_sentinel_normalized": True,
    "xpt_ibm_zero_sentinel_value": float(
        XPT_IBM_ZERO_SENTINEL
    ),
    "xpt_ibm_zero_sentinel_replacement_count": (
        total_sentinel_replacements
    ),
    "bridging_applied": False,
    "unit_conversion_applied": False,
    "complete_case_filter_applied": False,
    "imputation_applied": False,
    "phenotypic_age_calculated": False,
    "mortality_merged_into_biomarker_data": False,
    "final_scientific_results_allowed": False,
    "open_core_evidence_gaps": (
        CONFIG["governance"]["open_core_evidence_gaps"]
    ),
    "outputs": [
        str(path.relative_to(PROJECT_ROOT))
        for path in written_files
    ],
    "public_use_mortality_notice": (
        "For selected records, follow-up time or underlying cause of death "
        "may contain synthetic values for disclosure protection; vital "
        "status was not perturbed."
        if mortality_ingested else None
    ),
}

metadata_path = LOGS_ROOT / "01_data_ingestion_metadata.json"
metadata_path.write_text(
    json.dumps(metadata, indent=2, ensure_ascii=False),
    encoding="utf-8",
)
written_files.append(metadata_path)

print("Files written:")
for path in written_files:
    print(f"  - {path.relative_to(PROJECT_ROOT)}")


XPT IBM-zero sentinel replacements: 80902


,file,column,sentinel_value,replacement_count,replacement_value
0,DEMO_I.XPT,RIDAGEYR,5.397605e-79,396,0.0
1,DEMO_I.XPT,RIDAGEMN,5.397605e-79,31,0.0
2,DEMO_I.XPT,RIDEXAGM,5.397605e-79,10,0.0
3,DEMO_I.XPT,DMDEDUC3,5.397605e-79,239,0.0
4,DEMO_I.XPT,DMDHHSZA,5.397605e-79,6298,0.0
5,DEMO_I.XPT,DMDHHSZB,5.397605e-79,4715,0.0
6,DEMO_I.XPT,DMDHHSZE,5.397605e-79,7151,0.0
7,DEMO_I.XPT,WTMEC2YR,5.397605e-79,427,0.0
8,DEMO_I.XPT,INDFMPIR,5.397605e-79,92,0.0
9,BIOPRO_I.XPT,LBXSTB,5.397605e-79,27,0.0


Files written:
  - data\interim\nhanes_2015_2016_ingested_native.parquet
  - data\interim\nhanes_2017_2018_ingested_native.parquet
  - data\interim\nhanes_2015_2018_ingested_native.parquet
  - results\tables\01_component_ingestion_audit.csv
  - results\tables\01_merge_audit.csv
  - results\tables\01_variable_availability_audit.csv
  - results\tables\01_xpt_ibm_zero_sentinel_audit.csv
  - data\interim\nhanes_2015_2016_mortality_2019_public.parquet
  - data\interim\nhanes_2017_2018_mortality_2019_public.parquet
  - data\interim\nhanes_2015_2018_mortality_2019_public.parquet
  - results\tables\01_mortality_ingestion_audit.csv
  - results\tables\01_mortality_linkage_coverage_audit.csv
  - logs\01_data_ingestion_metadata.json


## 11. Reload verification

In [12]:
reloaded_biomarker = pd.read_parquet(
    INTERIM_ROOT
    / "nhanes_2015_2018_ingested_native.parquet"
)

assert len(reloaded_biomarker) == len(combined_biomarker)
assert list(reloaded_biomarker.columns) == list(
    combined_biomarker.columns
)
assert not reloaded_biomarker.duplicated(
    [cycle_column, "SEQN"]
).any()
assert reloaded_biomarker["LBXGLU"].equals(
    combined_biomarker["LBXGLU"]
)
assert "LBXSGL" not in reloaded_biomarker.columns

if mortality_ingested:
    reloaded_mortality = pd.read_parquet(
        INTERIM_ROOT
        / "nhanes_2015_2018_mortality_2019_public.parquet"
    )
    assert len(reloaded_mortality) == len(combined_mortality)
    assert not reloaded_mortality.duplicated(
        [cycle_column, "SEQN"]
    ).any()
    assert set(
        reloaded_mortality["MORTSTAT"].dropna().astype(int).unique()
    ).issubset({0, 1})

print("✅ Data ingestion completed and verified.")
print("Biomarker data remain in native NHANES units.")
print("Mortality data remain separate from the biomarker pipeline.")
print("Next notebook: 02_data_preprocessing.ipynb")


✅ Data ingestion completed and verified.
Biomarker data remain in native NHANES units.
Mortality data remain separate from the biomarker pipeline.
Next notebook: 02_data_preprocessing.ipynb
